# பாடம் 18: கிரிப்டோகிராபிக் ரசீத்களுடன் AI முகவரிகளை பாதுகாத்தல்

## கைமுறை நோக்ககம்

இந்த நோட்டுப்புத்தகம் நான்கு பயிற்சிகளைக் கொண்டு செல்கிறது:

1. முகவர் கருவி அழைப்புக்கான **உங்கள் முதலாவது ரசீதை கையொப்பமிடு**வும் அதை சரிபார்க்கவும்.
2. **ரசீதுடன் தற்காலிக மாற்றம் செய்யவும்** மற்றும் சரிபார்ப்பு தோல்வி அடையும் என்பதை காணவும்.
3. **மூன்று ரசீத்களின் சங்கிலியை கட்டடிக் கொள்ளவும்** மற்றும் சங்கிலி முழுமையை உறுதிப்படுத்தவும்.
4. **Microsoft Agent Framework கருவி அழைப்பை சுருளாக விழுங்கவும்** இதன் மூலம் ஒவ்வொரு செயலும் ஒரு ரசீதிடத்தை வெளிப்படுத்தும்.

அனைத்து கிரிப்டோகிராபிக் முதன்மையான பகுதிகள் நன்கு பராமரிக்கப்பட்ட நூலகங்களிலிருந்து இறக்குமதி செய்யப்படுகின்றன (`pynacl` Ed25519க்கான, `jcs` RFC 8785 canonical JSONக்கான, Python நிலையான நூலகம் `hashlib` SHA-256க்கான). ரசீதம் தானே சுருக்கமான Python ஆகும், நீங்கள் அதை வாசித்து திருத்த முடியும்.

செல்களை ஆர்டர் முறையில் இயக்கவும். ஒவ்வொரு பகுதியும் குறுகியதும் சுயமாக நிறைவாக உள்ளது.


## அமைப்பு

இரு சார்புகளையும் நிறுவவும். இரண்டுக்குமே அனுமதிப்புள்ளி உரிமங்கள் உள்ளன (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## உதவி பயன்பாடுகள்

இந்த இரு உதவிகள் படிபடியாக உள்ள பொருட்களின் base64url குறியாக்கம் (padding இல்லாமல்) மற்றும் SHA-256 ஹாஷிங் செய்கின்றன. அவை புத்தகக்குறிப்புயின் மீதியில் உள்ள ரசிது வாழி தானாக கவனிக்கின்றன.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## பகுதி 1: உங்கள் முதல் ரசீதிற்கு கையொப்பமிடுக

**Contoso Travel** இற்கான எங்கள் முகவர் சிட்னிட் இருந்து லாஸ் ஏஞ்சல்ஸ் வரை விமானங்களை ஒரு வாடிக்கையாளருக்காக தேடியிருக்கிறார் என்று கற்பனை செய்யுங்கள். எதிர்காலத்தில் ஒரு ஆடிட்டர் நம்மை நம்பாமலேயே அதை சரிபார்க்க, இந்த கருவி அழைப்பை கையொப்பமிட்ட ரசீதாக பதிவுசெய்ய விரும்புகின்றோம்.

### படி 1.1: கையொப்பம் இடும் விசையை உருவாக்குக

உற்பத்தியில், முகவரின் கையொப்ப விசை ஒரு ஹார்ட்வேர் பாதுகாப்பு மொட்யூல் (HSM), Azure விசை சுரங்கம் அல்லது அதே மாதிரியான பாதுகாக்கப்பட்ட சேமிப்பில் இருப்பார். இந்த பாடத்துக்காக நாம் நினைவகத்தில் புதிய விசையை உருவாக்குகிறோம்.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: ரசீது பेलோவை உருவாக்குதல்

பेलோவ் ரசீது உறுதிபடுத்த வேண்டிய அனைத்தையும் கொண்டுள்ளது: யார் செயல்பட்டார், எந்த கருவி, எந்த வாதங்களுடன், என்ன திரும்பியது, எந்த கொள்கையின் கீழ், மற்றும் எப்போது. ரசீது நுண்ணுணர்தல் உள்ளடக்கம் வெளிப்படாமல் இருக்கவோ, வாதங்களையும் முடிவுகளையும் நேரடியாக சேர்க்கவோாமல், அவற்றின் ஹாஷை எடுத்துக் கொள்கின்றோம்.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### படி 1.3: ரசீதை கையெழுத்திடவும் மற்றும் சேகரிக்கவும்

மூன்று படிகள்:

1. இரண்டு வேலைமுறைகளும் அதே தர்க்கரீதியான ரசீதத்தை உற்பத்தி செய்யும் போது, இரு வேலைமுறைகளும் பைட்-ஐடென்டிக்கல் பைட்களை உருவாக்க ஜிசிஎஸ் மூலம் பேலோடு canonicalize செய்யவும்.
2. canonical பைட்களை SHA-256 மூலம் ஹாஷ் செய்யவும்.
3. ஹாஷை Ed25519 தனியார் விசையுடன் கையெழுத்திடவும்.

கையொப்பம் பின்னர் இறுதி ரசீதை உருவாக்க முதன்மை பேலோட்டுடன் இணைக்கப்படுகிறது.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### படி 1.4: ரசீதியை சரிபார்க்கவும்

சரிபார்ப்பு செயல்முறை பின்னுக்கு மாற்றுகிறது. நாங்கள் கையொப்பத்தை அகற்றி, canonical ஹாஷை மீண்டும் கணக்கிட்டு, ரசீதியில் உள்ள பொது விசையுடன் கையொப்பத்தை பொருத்தி பார்க்கின்றோம்.

இந்தச் சரிபார்ப்பை செய்யும் ஆலோசகர் எங்களிடமிருந்து எந்தவிதமான உதவியும் தேவையில்லை, அவருக்கு தேவையானது ரசதி தான். எந்த சேவையையும் அழைக்கவேண்டியதில்லை, எந்த விசை அடைக்கலத்திற்கும் விசாரிக்க தேவையில்லை, எந்த நம்பிக்கையும் தேவைப்படமாட்டாது.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

நீங்கள் `Receipt is valid: True` என்று பார்க்க வேண்டும். முகவர் தனது முதல் குறியாக்க கையொப்பமிட்ட ஆய்வுக் குறிப்பு பதிவை உருவாக்கியுள்ளது.


## Section 2: ரசீது உருட்டல்

ரசீது முழுவதும் உருட்டு சான்று வழங்க வேண்டும். இதை நாம் நிரூபிப்போம்.

நாம் ரசீதின் ஒரு எழுத்தைத் திருத்தி சரிபார்ப்பு தோல்வி அடைவதை காண்போம்.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### என்ன ப்ள்ளுள்ளது?

நாம் `policy_id`ஐ மாற்றிய பொழுது, canonical bytes மாற்றப்பட்டது. அந்த bytes இன் SHA-256 ஹாஷ் மாற்றப்பட்டது. (அசல் ஹாஷ் மீது இருந்த) கையொப்பம் புதிய ஹாஷுடன் ஏற்றுக்கொள்ளவில்லை. சரிபார்ப்பு சத்தியமாக `False` ஐ திரும்ப அளிக்கிறது.

பெறுமானத்தில் எந்தத் துறையையும் மாற்றியும் சரிபார்க்கப்படும்படி செய்ய முடியாது, அத்தகைய நிலைக்கு அட்டாக்கர் தனிப்பட்ட விசையை கொண்டிருந்தால் மட்டுமே. தனிப்பட்ட விசை ஒரு விசை குவியலில் இருந்தாலும் பொதுவான விசை வெளியிடப்பட்டாலும், குறுக்கலை மறைக்க முடியாது.

நீங்களே முயற்சி செய்யுங்கள்: மேலே உள்ள செலிலுள்ள `tool_name` அல்லது `agent_id` அல்லது `timestamp` ஐ மாற்றி மீண்டும் இயக்குங்கள். ஒவ்வொரு மாற்றத்திலும் ஒரு தவறான பெறுமதி உண்டாகும்.


## பகுதி 3: ரசீத்களை ஒன்றாகச் சங்கிலியாக்குதல்

ஒன்றே ஒரு செயலுக்கு ஒரு ரசீத்கத்தைப் பாதுகாக்கிறது. பெரும்பாலான முகவர்கள் பல நடவடிக்கைகள் எடுப்பார்கள். முழு தொடர் தகர்க்கத் தெரியும்படி இருக்க, ஒவ்வொரு ரசீதையும் அதன் முன்பு உள்ள ரசீதுடன் இணைக்கிறோம், அதாவது புதிய ரசீதின் தரவுப் பகுதியில் முன்பு உள்ள ரசீதின் ஹாஷ் சேர்க்கப்படுகிறது.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

யாராவது ஒரு ரசீதைக் களைத்தாலும் அல்லது மறுசீரமைத்தாலும், சங்கிலி அதே இடத்தில் துண்டிக்கிறது. பின்னர் எந்தவொரு ரசீதின் உறுதிப்படுத்தலும் தோல்வியடைகிறது ஏனென்றால் அதன் `previous_receipt_hash` அதன் முன்னோடியின் உண்மையான ஹாஷுடன் பொருந்தாது.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

இப்போது நடுவில் உள்ள ரசீதை மாற்றி சங்கிலியை உடைக்கவும் மற்றும் மீண்டும் சரிபார்க்கவும். மாற்றப்பட்ட ரசீது அதன் கையொப்பச் சோதனையில் தோல்வியடைகிறது, மேலும் அடுத்த ரசீது அதன் சங்கிலி-இணைப்பு சோதனையில் தோல்வியடைகிறது (ஏனெனில் அதின் `previous_receipt_hash` இனிமேல் மாற்றப்பட்ட நடுவை ரசீதின் ஹாஷுடன் பொருந்தவில்லை).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ரசீது 0 இன்னும் சரிபார்க்கப்படுகிறது (இது மாற்றப்படவில்லை மற்றும் சார்ந்த முன் ஒன்றுமில்லை). ரசீது 1 தனது கையொப்ப சோதனையில் தோல்வியடைகிறது ஏனெனில் நாம் `tool_args_hash` ஐ மாற்றினோம். ரசீது 2 தனது சங்கிலி-இணைப்பு சோதனையில் தோல்வியடைகிறது ஏனெனில் அதன் `previous_receipt_hash` அசல் (இப்போது மாற்றப்பட்ட) ரசீது 1 ஐ எதிர்கொண்டு கணக்கிடப்பட்டது.

ஒரு தாக்குபவர் மாற்றிய ரசீது 1 ஐ மீண்டும் கையொப்பமிடினாலும் (அவர்கள் தனிப்பட்ட விசையுடன் இல்லாமல் இதை செய்ய முடியாது), ரசீது 2 இல் சங்கிலி-இணைப்பு பொருத்தமின்மை இன்னும் தடுமாறலை வெளிப்படுத்தும். மாற்றத்தை மறைக்க, தாக்குபவர் மாற்றம் தொடங்கும் இடத்திலிருந்து ஒவ்வொரு ரசீதையும் மீண்டும் கையொப்பமிட வேண்டியிருக்கும், இதற்கு தனிப்பட்ட விசை தேவையாகும்.


## பகுதி 4: ஒரு முகவர் கருவி அழைப்பை ரசீதுக்கான கையொப்பத்துடன் மூடு

ஒரு உண்மையான நியமனதில், ஒவ்வொரு முகவர் ஆசிரியரும் `make_receipt` ஐ அழைக்க நினைத்துக் கொள்வதை நீங்கள் அறிய விரும்ப மாட்டீர்கள். ஒவ்வொரு கருவி அழைப்புக்கும் ரசீதுக்கான கையொப்பம் தானாகவே ஏற்பட வேண்டும்.

இங்கே மிக எளிய மாதிரி உள்ளது: எந்த அழைக்கக்கூடிய கருவி செயல்பாட்டையும் எடுத்துக்கொண்டு அதற்கான ரசீதினை வெளியிடும் பதிப்பை திருப்பித் தரும் ஒரு இறுகிய வகுப்பு. இது Microsoft Agent Framework (`agent_framework.azure`) உட்பட எந்த முகவர் கட்டமைப்புக்கும் ஏற்றுக்கொள்ள முடியும்.

நீங்கள் Azure AI Foundry திட்டத்தை அமைக்கவில்லை என்றால் கூட, கீழே உள்ள உள்ளூர் நிறுவல் மாதிரி இந்த மாதிரியை விளக்குகின்றது.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework உடன் ஒருங்கிணைத்தல்

மேலே காட்டிய `ReceiptedTool` ஓரத்தை எந்த Framework-ஐயும் சாராதவையாக உள்ளது. Microsoft Agent Framework-ஆல் கட்டப்பட்ட ஒரு முகவரியின் உள்ளே இதைப் பயன்படுத்த, மூடியுள்ள செயல்பாட்டை ஒரு கருவியாக பதிவு செய்ய வேண்டும். ஒரு வரைவை (நீங்கள் mock-ஐ உண்மையான Azure AI Foundry கருவி பதிவுசெய்தலுடன் மாற்றுவீர்கள்):

```python
# ஒருங்கிணைப்பு வடிவத்தை காட்டும் படுகோள் குறியீடு.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="நீங்கள் ஒரு Contoso பயண முகவர்...",
#     tools=[receipted_lookup],   # மூடிய கருவி, நேரடி செயல்பாடு அல்ல
# )
# response = agent.run("சிட்னியிடமிருந்து லாஸ் ஏஞ்சலஸுக்கு ஜூன் மாதத்தில் விமானங்கள் காண்க.")
#
# # ஓட்டத்திற்குப் பிறகு, முகவர் செய்த ஒவ்வொரு கருவி அழைப்புக்கும் ஒரு கையெழுத்திடப்பட்ட ரசீது உண்டு:
# audit_chain = receipted_lookup.receipts
```

முகவர் Framework-க்கு எதிர்பார்ப்பு பற்றிய எந்த தகவலும் தேவையில்லை. எதிர்பார்ப்பு கையொப்பம் கருவிக்குப் பிறக்கப்பட்டுள்ளதா, framework-க்கு இணைக்கப்படவில்லை. இது முகவரியை மறுசரிசெய்யாமல் இருந்து தற்போதைய முகவர் குறியீட்டில் மூலத்தைச் சேர்ப்பதற்கான வழி.


## மீளாய்வு மற்றும் விரிவாக்க சவால்

உங்களுக்கு:

- Ed25519 விசை ஜோடியை உருவாக்கியுள்ளீர்கள்.
- ஒரு முகவர் கருவி அழைப்புக்கான ரசீதை உருவாக்கி கையொப்பமிட்டுள்ளீர்கள்.
- பொதுக்குறியீட்டை மட்டுமே பயன்படுத்தி ஆஃப்லைனில் ரசீதலை உறுதிப்படுத்தியுள்ளீர்கள்.
- ஒரு ரசீதலை மாற்றி அதன்பிறகு உறுதிப்படுத்தல் தோல்வியடைந்ததை 관찰ித்தீர்கள்.
- மூன்று ரசீத்களைக் கொண்ட ஹாஷ்-செயினாட்டு தொடரை கட்டியுள்ளீர்கள்.
- சங்கிலியின் நடு பகுதியை மாற்றி கையொப்ப தோல்வியும் சங்கிலி இணைப்பு தோல்வியும் 관찰ித்தீர்கள்.
- ஒரு கருவி செயல்பாட்டை தானாக ரசீத அமைப்புடன் அடையாளம் காட்டியுள்ளீர்கள்.

**விரிவாக்க சவால்.** விநியோக சோதனைக்கான UUID-வுமான `request_id` என்ற விருப்பத்தை ரசீத அமைப்பில் சேர்க்கவும். `make_receipt` இல் இதை சேர்த்து, ரசீத்கள் முழுமையாக உறுதிப்படுத்தப்படுகிறதா என உறுதிப்படுத்தவும். பிறகு கையொப்பமிட்ட பிறகு அந்த களத்தை மாற்றி உறுதிப்படுத்தல் தோல்வியடைவதை உறுதிசெய்க. இவ்வாறு canonical குறியீட்டின் ஒவ்வொரு பைடும் கையொப்பத்தில் எப்படி பங்காற்றுகின்றது என்பதை உங்களால் உள்ளூடாகக் கற்றுக்கொள்ள முடியும்.

**முக்கிய எல்லை.** ரசீதுகள் மூன்று விஷயங்களை மட்டும் நிரூபிக்கின்றன: ஒப்பந்தம் (இந்த விசை இந்த உள்ளடக்கத்திற்கு கையொப்பமிட்டது), ஒருங்கிணைப்பு (கையொப்பமிடும் போது உள்ளடக்கம் மாற்றப்படவில்லை), ஆணையிடுதல் (இந்த ரசீது அந்த ரசீதுக்குப் பிறகு வந்தது). அவை முகவரியின் செயல்பாடு சரியானதா, `policy_id` யில் குறிப்பிடப்பட்ட கொள்கை உண்மையில் மதிப்பாய்வு செய்யப்பட்டதா, அல்லது முகவர் ஒவ்வொரு விதிமுறையையும் பின்பற்றினாரா என்பதை நிரூபிக்கவில்லை. ரசீதுகள் ஒரு அடித்தளம். நிர்வாகம் நீங்கள் அதை மேல் கட்டும் அமைப்புதான்.

இந்த எல்லையை மனதில் வைத்து பாடம் README-வை மீண்டும் வாசியுங்கள். ரசீத்களுடன் கூடிய குழுக்களின் மிகப் பொதுவான தவறு "நமக்கு ரசீதுகள் உள்ளன" என்றால் "நாம் நிர்வகிக்கப்படுகிறோம்" என்று கருதுவதாகும். அது தவறு. ரசீதுகள் முகவர் நடத்தையை சோதனை செய்யக்கூடியதாக ஆக்கும். அவை அதை சரியானதாக ஆக்குவதில்லை.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
